# Materials Project Formation Energy Distribution

MP has a curious bimodality in its formation energies. Considering the formation energies are the result of a carefully fitted correction scheme (see `pymatgen.entries.compatibility.MaterialsProject2020Compatibility`) that takes into account only the composition of a compound, let's look at which elements dominate the upper and lower modes.

In [ ]:
# mp-api: Materials Project data
%pip install pymatviz mp-api

In [ ]:
import pandas as pd
from mp_api.client import MPRester

from pymatviz import BarPlotWidget, count_elements, histogram, ptable_heatmap


__author__ = "Janosh Riebesell"
__date__ = "2022-08-11"

Check if all of MP also has a bi-modal formation energy distribution. Short answer: yes it does.

In [ ]:
# MPRester reads your API key from the MP_API_KEY environment variable
with MPRester(use_document_model=False) as mpr:
    e_form_all_mp = mpr.materials.summary.search(
        fields=["material_id", "formula_pretty", "formation_energy_per_atom"]
    )

df_e_form_all_mp = pd.DataFrame(map(dict, e_form_all_mp)).set_index("material_id")

In [ ]:
# cache MP data
# %store df_e_form_all_mp

# load cached MP data
# %store -r df_e_form_all_mp

In [ ]:
# formation energy distribution of all MP entries
e_form_valley = -1.35
widget = histogram(
    df_e_form_all_mp["formation_energy_per_atom"],
    bins=200,
    x_range=(-5, 3),
    x_axis={"label": "Formation energy [eV/atom]"},
)
widget.ref_lines = [
    {
        "type": "vertical",
        "x": e_form_valley,
        "style": {"color": "orange", "dash": "6 4"},
        "annotation": {"text": f"{e_form_valley} eV/atom", "color": "orange"},
    }
]
widget

First, let's plot elemental prevalence as a heatmap over the periodic table for things above and below `e_form_valley`.

In [ ]:
for comparator, label in ((">", "higher"), ("<=", "lower")):
    title = f"{label.title()} mode {comparator} {e_form_valley} eV/atom"
    ptable_heatmap(
        df_e_form_all_mp.query(
            f"formation_energy_per_atom {comparator} {e_form_valley}"
        ).formula_pretty,
        color_bar={"title": title},
    ).show()

Looks like the lower mode is mostly oxides, whereas the higher mode is more diverse also containing many nitrides, sulfides and selenides.

Another way to visualize this are bar charts.

In [ ]:
for comparator, label in ((">", "higher"), ("<=", "lower")):
    elem_counts = count_elements(
        df_e_form_all_mp.query(
            f"formation_energy_per_atom {comparator} {e_form_valley}"
        ).formula_pretty
    )
    top_counts = elem_counts.nlargest(20)
    title = f"{label.title()} mode {comparator} {e_form_valley} eV/atom"
    BarPlotWidget(
        [{"x": top_counts.index.astype(str).to_list(), "y": top_counts}],
        x_axis={"label": title},
        y_axis={"label": "Element count"},
    ).show()

Note that this significant lowering of oxide formation energies compared to other anions might at least partially be an artifact of too little experimental data outside oxide systems. In other words, perhaps there should be stronger corrections applied to nitrides, selenides, etc. as well but because there's insufficient experimental data to fit a robust correction scheme there, MP doesn't.